# Marmousi2 Acoustic bv1.2 Inversion Validation

This notebook validates the inversion side of the Marmousi2 acoustic case using synthetic-true observations. Parameters, model roles, survey subset, wavelet setup, and FWI execution are defined directly in the notebook. No CLI runner is used.

## 1. Paths And Imports

In [ ]:
from __future__ import annotations

import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ADFWI
from ADFWI.model import AcousticModel
from ADFWI.propagator import AcousticPropagator, GradProcessor
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import wavelet

CASE_DIR = REPO_ROOT / "examples" / "acoustic" / "01-model-test" / "01-Marmousi2"
VALIDATION_DIR = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12"
OUTPUT_ROOT = VALIDATION_DIR / "outputs"

CASE_DIR

from ADFWI.fwi import AcousticFWI
from ADFWI.fwi.misfit import Misfit_waveform_L2


## 2. Local Case Definitions

In [ ]:
def load_npz(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"required Marmousi2 file does not exist: {path}")
    return np.load(path, allow_pickle=True)


def optional_bound(value):
    arr = np.asarray(value)
    if arr.dtype == object:
        values = arr.tolist()
        if values is None or any(item is None for item in values):
            return None
    if arr.size != 2:
        return None
    return float(arr[0]), float(arr[1])


def cumulative_trapezoid(values: np.ndarray, dt: float) -> np.ndarray:
    out = np.zeros_like(values, dtype=np.float32)
    if values.size > 1:
        out[1:] = np.cumsum((values[:-1] + values[1:]) * (0.5 * dt), dtype=np.float64).astype(np.float32)
    return out


def build_source_wavelet(nt: int, dt: float, f0: float) -> np.ndarray:
    _, src_v = wavelet(nt, dt, f0, amp0=1)
    return cumulative_trapezoid(src_v.astype(np.float32), dt)


def build_survey(obs_npz, f0: float, *, shot_count: int | None = None, nt_samples: int | None = None) -> Survey:
    full_nt = int(obs_npz["nt"])
    nt = full_nt if nt_samples is None else int(nt_samples)
    dt = float(obs_npz["dt"])
    src_loc = np.asarray(obs_npz["src_loc"], dtype=np.int64)
    src_type = np.asarray(obs_npz["src_type"]).astype(str)
    if shot_count is not None:
        src_loc = src_loc[:shot_count]
        src_type = src_type[:shot_count]
    rcv_loc = np.asarray(obs_npz["rcv_loc"], dtype=np.int64)
    rcv_type = np.asarray(obs_npz["rcv_type"]).astype(str)

    src_v = build_source_wavelet(nt, dt, f0)
    source = Source(nt=nt, dt=dt, f0=f0)
    for (src_x, src_z), src_kind in zip(src_loc, src_type):
        source.add_source(int(src_x), int(src_z), src_v, src_type=str(src_kind))

    receiver = Receiver(nt=nt, dt=dt)
    for (rcv_x, rcv_z), rcv_kind in zip(rcv_loc, rcv_type):
        receiver.add_receiver(int(rcv_x), int(rcv_z), rcv_type=str(rcv_kind))
    return Survey(source, receiver)


def build_acoustic_model(model_npz, *, vp_grad: bool, auto_update_rho: bool, abc_type: str = "PML") -> AcousticModel:
    return AcousticModel(
        float(model_npz["ox"]),
        float(model_npz["oz"]),
        int(model_npz["nx"]),
        int(model_npz["nz"]),
        float(model_npz["dx"]),
        float(model_npz["dz"]),
        np.asarray(model_npz["vp"], dtype=np.float32),
        np.asarray(model_npz["rho"], dtype=np.float32),
        vp_bound=optional_bound(model_npz["vp_bound"]),
        rho_bound=optional_bound(model_npz["rho_bound"]),
        vp_grad=vp_grad,
        rho_grad=False,
        auto_update_rho=auto_update_rho,
        free_surface=bool(model_npz["free_surface"]),
        abc_type=abc_type,
        abc_jerjan_alpha=0.007,
        nabc=int(model_npz["nabc"]),
    )


def tensor_summary(value: torch.Tensor) -> dict[str, object]:
    detached = value.detach()
    return {
        "shape": list(detached.shape),
        "device": str(detached.device),
        "dtype": str(detached.dtype).replace("torch.", ""),
        "finite": bool(torch.isfinite(detached).all().item()),
        "min": float(detached.min().cpu().item()),
        "max": float(detached.max().cpu().item()),
        "norm": float(torch.linalg.norm(detached.reshape(-1)).cpu().item()),
    }


## 3. Parameter Definitions

In [ ]:
INVERSION_CONFIG = {
    "device": "npu:0",
    "dtype": "float32",
    "fallback_cpu": False,
    "true_model_file": "true_model.npz",
    "inversion_model_file": "init_model.npz",
    "f0": 5.0,
    "shots": 3,
    "nt_samples": 3000,
    "checkpoint_segments": 10,
    "iterations_short": 10,
    "iterations_long": 100,
    "optimizer": "adam",
    "lr": 10.0,
    "scheduler_step_size": 200,
    "scheduler_gamma": 0.75,
    "misfit": "legacy-l2",
    "waveform_normalize": True,
    "auto_update_rho": True,
    "grad_mute_top": 12,
    "gradient_processor": "legacy",
    "abc_type": "PML",
}
INVERSION_CONFIG


## 4. Backend Setup

In [ ]:
backend = ADFWI.set_backend(
    INVERSION_CONFIG["device"],
    dtype=INVERSION_CONFIG["dtype"],
    fallback=INVERSION_CONFIG["fallback_cpu"],
    prefer=("npu", "cpu"),
)
ADFWI.backend_diagnostics()


## 5. Model Definitions

In [ ]:
true_model_npz = load_npz(CASE_DIR / "data" / "model" / INVERSION_CONFIG["true_model_file"])
init_model_npz = load_npz(CASE_DIR / "data" / "model" / INVERSION_CONFIG["inversion_model_file"])
true_model = build_acoustic_model(
    true_model_npz,
    vp_grad=False,
    auto_update_rho=False,
    abc_type=INVERSION_CONFIG["abc_type"],
)
model = build_acoustic_model(
    init_model_npz,
    vp_grad=True,
    auto_update_rho=INVERSION_CONFIG["auto_update_rho"],
    abc_type=INVERSION_CONFIG["abc_type"],
)
initial_vp = model.vp.detach().clone()
model_roles = {
    "true_model": {"file": INVERSION_CONFIG["true_model_file"], "vp": tensor_summary(true_model.vp)},
    "initial_model": {"file": INVERSION_CONFIG["inversion_model_file"], "vp": tensor_summary(model.vp)},
}
model_roles


## 6. Observation System And Wavelet Definition

In [ ]:
obs_npz = load_npz(CASE_DIR / "data" / "waveform" / "obs_data.npz")
survey = build_survey(
    obs_npz,
    f0=INVERSION_CONFIG["f0"],
    shot_count=INVERSION_CONFIG["shots"],
    nt_samples=INVERSION_CONFIG["nt_samples"],
)
source_wavelet = build_source_wavelet(survey.source.nt, survey.source.dt, INVERSION_CONFIG["f0"])
survey_summary = {
    "shots": survey.source.num,
    "receivers": survey.receiver.num,
    "nt": survey.source.nt,
    "dt": survey.source.dt,
    "f0": INVERSION_CONFIG["f0"],
    "wavelet_norm": float(np.linalg.norm(source_wavelet)),
    "src_x_range": [int(np.min(survey.source.get_loc()[:, 0])), int(np.max(survey.source.get_loc()[:, 0]))],
    "rcv_x_range": [int(np.min(survey.receiver.get_loc()[:, 0])), int(np.max(survey.receiver.get_loc()[:, 0]))],
}
survey_summary


## 7. Generate Synthetic-True Observed Data

In [ ]:
RUN_SYNTHETIC_OBS = False

if RUN_SYNTHETIC_OBS:
    true_propagator = AcousticPropagator(true_model, survey)
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        observed_record = true_propagator.forward(
            shot_index=np.arange(INVERSION_CONFIG["shots"]),
            checkpoint_segments=INVERSION_CONFIG["checkpoint_segments"],
        )
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    observed_seconds = time.perf_counter() - start
    obs_data = SeismicData(survey)
    obs_data.record_data(observed_record)
    observed_summary = {
        "source": "synthetic-true",
        "seconds": observed_seconds,
        "pressure": tensor_summary(observed_record["p"]),
    }
else:
    obs_data = None
    observed_summary = {"status": "skipped", "reason": "set RUN_SYNTHETIC_OBS=True"}

observed_summary


## 8. FWI Definition

In [ ]:
def build_fwi(obs_data: SeismicData):
    propagator = AcousticPropagator(model, survey)
    grad_mask = np.ones((model.nz, model.nx), dtype=np.float32)
    grad_mask[: INVERSION_CONFIG["grad_mute_top"], :] = 0.0
    gradient_processor = GradProcessor(grad_mask=grad_mask)
    optimizer = torch.optim.Adam(model.parameters(), lr=INVERSION_CONFIG["lr"])
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=INVERSION_CONFIG["scheduler_step_size"],
        gamma=INVERSION_CONFIG["scheduler_gamma"],
    )
    loss_fn = Misfit_waveform_L2(dt=1.0)
    return AcousticFWI(
        propagator,
        model,
        optimizer,
        scheduler,
        loss_fn,
        obs_data,
        gradient_processor=gradient_processor,
        waveform_normalize=INVERSION_CONFIG["waveform_normalize"],
        cache_result=True,
        cache_result_epoch=1,
        save_fig_epoch=-1,
    )

"FWI builder ready"


## 9. Run 10-Iteration Inversion

Set `RUN_SYNTHETIC_OBS = True` in the previous cell and `RUN_INVERSION_10 = True` here to run the short validation.

In [ ]:
RUN_INVERSION_10 = False

if RUN_INVERSION_10:
    if obs_data is None:
        raise RuntimeError("Run the synthetic-observed-data cell first with RUN_SYNTHETIC_OBS=True")
    fwi = build_fwi(obs_data)
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    start = time.perf_counter()
    fwi.forward(
        iteration=INVERSION_CONFIG["iterations_short"],
        batch_size=INVERSION_CONFIG["shots"],
        checkpoint_segments=INVERSION_CONFIG["checkpoint_segments"],
    )
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    inversion_seconds = time.perf_counter() - start
    losses = [float(value) for value in fwi.iter_loss]
    inversion10_summary = {
        "iterations": INVERSION_CONFIG["iterations_short"],
        "seconds": inversion_seconds,
        "initial_loss": losses[0],
        "final_loss": losses[-1],
        "loss_delta": losses[-1] - losses[0],
        "vp_update_norm": float(torch.linalg.norm((model.vp.detach() - initial_vp).reshape(-1)).cpu().item()),
    }
else:
    fwi = None
    inversion10_summary = {"status": "skipped", "reason": "set RUN_INVERSION_10=True"}

inversion10_summary


## 10. Optional 100-Iteration Inversion

Only run after the 10-iteration result is reasonable. Rebuild the notebook state from the top before running a long test.

In [ ]:
RUN_INVERSION_100 = False

if RUN_INVERSION_100:
    if obs_data is None:
        raise RuntimeError("Run the synthetic-observed-data cell first with RUN_SYNTHETIC_OBS=True")
    fwi = build_fwi(obs_data)
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    start = time.perf_counter()
    fwi.forward(
        iteration=INVERSION_CONFIG["iterations_long"],
        batch_size=INVERSION_CONFIG["shots"],
        checkpoint_segments=INVERSION_CONFIG["checkpoint_segments"],
    )
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    inversion100_summary = {
        "iterations": INVERSION_CONFIG["iterations_long"],
        "seconds": time.perf_counter() - start,
        "final_loss": float(fwi.iter_loss[-1]),
    }
else:
    inversion100_summary = {"status": "skipped", "reason": "set RUN_INVERSION_100=True"}

inversion100_summary
